# 07b -- OOF probabilities recovery for Joint Hurdle BiLSTM v4

Этот notebook существует отдельно от основного `07_LSTM.ipynb`.

Цель -- получить честные per-user OOF predictions на **глобально выбранной `BEST_EPOCH`**:

```text
user_id
cutoff_date
y_true
target_nonzero
pred_log
gate_prob
positive_log
direct_log
hurdle_log
```

Основной LSTM run намеренно не меняется.

## Что notebook умеет переиспользовать

Для каждого temporal fold порядок такой:

```text
1. Есть готовый prediction_epoch_BEST.npz из основного CV?
      -> использовать без обучения.

2. Есть уже восстановленный prediction из этого notebook?
      -> использовать.

3. Original resume.pt находится ровно на BEST_EPOCH?
      -> загрузить веса и только сделать inference.

4. Original/recovery resume находится раньше BEST_EPOCH?
      -> продолжить обучение до BEST_EPOCH.

5. Resume уже ушел дальше BEST_EPOCH?
      -> назад веса не отмотать, поэтому retrain только этого fold
         с нуля до BEST_EPOCH.
```

В завершенном основном run January predictions уже сохранялись по каждой эпохе, поэтому January должен переиспользоваться без retrain.

Для Nov/Dec, если exact selected-epoch weights не сохранились, потребуется retrain **только до `BEST_EPOCH=10`**, а не полный early-stopping run до 20/12 эпох.

## Важная оговорка

Если потерянный fold изначально обучался на MPS, а recovery запускается на CUDA, восстановленная модель остается честной OOF-моделью той же архитектуры и training policy, но ее predictions не обязаны быть бит-в-бит равны потерянным predictions исходного run.

После recovery notebook сравнивает новый fold RMSLE с `cv_history.csv`, чтобы было видно величину расхождения.

## 1. Setup и те же данные/гиперпараметры

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import hashlib
import json
import math
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def find_project_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data" / "lstm" / "meta.json").exists():
            return path
    raise FileNotFoundError(
        "Не найден data/lstm/meta.json. Сначала запустите обновленный 06_LSTM_Data_Preparation.ipynb."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_expanded_es"
REFERENCE_MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_robust"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

if META.get("format_version") != 2:
    raise RuntimeError(
        "LSTM v4 ожидает data format v2. Перезапустите обновленный 06_LSTM_Data_Preparation.ipynb."
    )

LABELED_CUTOFFS = META["labeled_cutoffs"]
INFERENCE_CUTOFF = META["inference_cutoff"]
BASE_SEQUENCE_FEATURES = META["base_sequence_features"]
CALENDAR_FEATURES = META["calendar_sequence_features"]
RAW_STATIC_FEATURES = META["static_features"]
RAW_STATIC_LOG_FEATURES = META["static_log_copy_features"]
EXTRA_STATIC_FEATURES = set(META.get("extra_static_features", []))
SEQ_LEN = int(META["seq_len"])
N_USERS = int(META["user_count"])

BASE_INDEX = {name: i for i, name in enumerate(BASE_SEQUENCE_FEATURES)}

# Флаги меняю по одному после baseline run.
USE_USER_EMBEDDING = False
USER_EMBED_DIM = 4
USE_CONV_STEM = False

USE_ABSOLUTE_TIME = True
USE_FUTURE_CALENDAR_STATIC = False
USE_CUTOFF_SEASONAL_STATIC = False

BATCH_SIZE = 32768  # 768
MAX_CV_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MIN_DELTA = 1e-4
MIN_CV_EPOCHS_BEFORE_STOP = 8
N_CV_FOLDS = 3
LR = 3.5e-4
EMBED_LR = 1.0e-4
WEIGHT_DECAY = 1.2e-3
EMBED_WEIGHT_DECAY = 8e-3
RANDOM_STATE = 42
FINAL_SEEDS = [42, 143, 2026]

AUX_GATE_WEIGHT = 0.03
AUX_POS_WEIGHT = 0.10
AUX_DIRECT_WEIGHT = 0.05

REFERENCE_MEAN_CV_RMSLE = 1.715288
REFERENCE_JAN_HOLDOUT_RMSLE = 1.675888
AUTO_SKIP_FINAL_IF_WEAK = True
MAX_MEAN_CV_DEGRADATION_FOR_FINAL = 0.003
MAX_JAN_DEGRADATION_FOR_FINAL = 0.006
FORCE_FINAL_TRAIN = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
NUM_WORKERS = 4 if DEVICE.type == "cuda" else 0
USE_AMP = DEVICE.type == "cuda"


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_static_features():
    selected = []
    for name in RAW_STATIC_FEATURES:
        if name not in EXTRA_STATIC_FEATURES:
            selected.append(name)
            continue

        if name == "cutoff_time_years" and USE_ABSOLUTE_TIME:
            selected.append(name)
        elif name.startswith("forecast_") and USE_FUTURE_CALENDAR_STATIC:
            selected.append(name)
        elif (
            name.startswith("cutoff_")
            and name != "cutoff_time_years"
            and USE_CUTOFF_SEASONAL_STATIC
        ):
            selected.append(name)
    return selected


STATIC_FEATURES = choose_static_features()
STATIC_INDICES = np.array(
    [RAW_STATIC_FEATURES.index(name) for name in STATIC_FEATURES],
    dtype=np.int64,
)
STATIC_LOG_FEATURES = [name for name in RAW_STATIC_LOG_FEATURES if name in STATIC_FEATURES]
STATIC_LOG_INDICES = [STATIC_FEATURES.index(name) for name in STATIC_LOG_FEATURES]

set_seed()
print("device:", DEVICE)
print("AMP:", USE_AMP)
print("labeled cutoffs:", LABELED_CUTOFFS)
print("CV validation cutoffs:", LABELED_CUTOFFS[-N_CV_FOLDS:])
print("inference:", INFERENCE_CUTOFF)
print("users:", N_USERS)
print("static features:", len(STATIC_FEATURES), "/", len(RAW_STATIC_FEATURES))
print("user embedding:", USE_USER_EMBEDDING)
print("future calendar static:", USE_FUTURE_CALENDAR_STATIC)
print("max CV epochs:", MAX_CV_EPOCHS)
print("early stopping patience:", EARLY_STOPPING_PATIENCE)
print("early stopping min delta:", EARLY_STOPPING_MIN_DELTA)
print("minimum epochs before stop:", MIN_CV_EPOCHS_BEFORE_STOP)
print("model dir:", MODEL_DIR)

device: cuda
AMP: True
labeled cutoffs: ['2025-04-19', '2025-05-19', '2025-06-18', '2025-07-18', '2025-08-17', '2025-09-16', '2025-10-16', '2025-11-15', '2025-12-15', '2026-01-14']
CV validation cutoffs: ['2025-11-15', '2025-12-15', '2026-01-14']
inference: 2026-02-13
users: 250000
static features: 92 / 103
user embedding: False
future calendar static: False
max CV epochs: 30
early stopping patience: 8
early stopping min delta: 0.0001
minimum epochs before stop: 8
model dir: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/models/lstm_hurdle_v4_expanded_es


## 2. Dataset

In [4]:
def known_user_mask_from_cutoffs(cutoffs):
    mask = np.zeros(N_USERS, dtype=np.bool_)
    for cutoff in cutoffs:
        idx = np.load(DATA_DIR / cutoff / "user_index.npy", mmap_mode="r")
        mask[np.asarray(idx, dtype=np.int64)] = True
    return mask


class HybridDataset(Dataset):
    def __init__(self, cutoff, with_target=True, known_user_mask=None):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.static = np.load(path / "static.npy", mmap_mode="r")
        self.calendar = np.load(path / "calendar.npy", mmap_mode="r").astype(np.float32)
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.user_index = np.load(path / "user_index.npy", mmap_mode="r")
        self.history_length = np.load(path / "history_length.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None
        self.known_user_mask = known_user_mask

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        sequence = np.concatenate([
            np.asarray(self.X[i], dtype=np.float32),
            self.calendar,
        ], axis=1)

        user_index = int(self.user_index[i])
        user_known = True if self.known_user_mask is None else bool(self.known_user_mask[user_index])

        raw_static = np.asarray(self.static[i], dtype=np.float32)
        static = raw_static[STATIC_INDICES]

        result = (
            torch.from_numpy(sequence),
            torch.from_numpy(static),
            torch.tensor(user_index, dtype=torch.long),
            torch.tensor(user_known, dtype=torch.bool),
            torch.tensor(int(self.history_length[i]), dtype=torch.long),
        )

        if self.y is None:
            return result

        return (*result, torch.tensor(float(self.y[i]), dtype=torch.float32))


def make_loader(
    cutoffs,
    shuffle=False,
    with_target=True,
    batch_size=BATCH_SIZE,
    known_user_mask=None,
):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]

    dataset = ConcatDataset([
        HybridDataset(
            cutoff,
            with_target=with_target,
            known_user_mask=known_user_mask,
        )
        for cutoff in cutoffs
    ])

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )

## 3. Sequence и summary features

In [5]:
def history_mask_from_lengths(lengths, steps=SEQ_LEN):
    positions = torch.arange(steps, device=lengths.device).unsqueeze(0)
    starts = steps - lengths.unsqueeze(1)
    return positions >= starts


def causal_masked_mean(values, valid_mask, window):
    values = values * valid_mask
    num = F.avg_pool1d(
        F.pad(values.unsqueeze(1), (window - 1, 0)),
        window,
        stride=1,
    ).squeeze(1) * window
    den = F.avg_pool1d(
        F.pad(valid_mask.unsqueeze(1), (window - 1, 0)),
        window,
        stride=1,
    ).squeeze(1) * window
    return num / den.clamp_min(1.0)


def first_difference(values, valid_mask):
    previous_valid = F.pad(valid_mask[:, :-1], (1, 0))
    both_valid = valid_mask * previous_valid
    diff = F.pad(values[:, 1:] - values[:, :-1], (1, 0))
    return diff * both_valid


def safe_ratio(numerator, denominator, max_value=5.0):
    ratio = numerator / denominator.clamp_min(1e-3)
    ratio = torch.where(denominator > 0, ratio, torch.zeros_like(ratio))
    return ratio.clamp(0, max_value)


def make_sequence_features(x, lengths):
    steps = x.shape[1]
    valid_mask = history_mask_from_lengths(lengths, steps).float()

    searches_log = x[..., BASE_INDEX["searches"]]
    search_to_cart_log = x[..., BASE_INDEX["search_to_cart"]]
    search_to_ord_log = x[..., BASE_INDEX["search_to_ord"]]
    cat_to_cart_log = x[..., BASE_INDEX["cat_to_cart"]]
    cat_to_ord_log = x[..., BASE_INDEX["cat_to_ord"]]
    to_cart_log = x[..., BASE_INDEX["to_cart"]]
    to_ord_log = x[..., BASE_INDEX["to_ord"]]
    gmv_search_log = x[..., BASE_INDEX["gmv_search"]]
    gmv_log = x[..., BASE_INDEX["gmv"]]
    active = x[..., BASE_INDEX["active"]]

    searches = torch.expm1(searches_log).clamp_min(0)
    search_to_cart = torch.expm1(search_to_cart_log).clamp_min(0)
    search_to_ord = torch.expm1(search_to_ord_log).clamp_min(0)
    cat_to_cart = torch.expm1(cat_to_cart_log).clamp_min(0)
    cat_to_ord = torch.expm1(cat_to_ord_log).clamp_min(0)
    to_cart = torch.expm1(to_cart_log).clamp_min(0)
    to_ord = torch.expm1(to_ord_log).clamp_min(0)
    gmv_search = torch.expm1(gmv_search_log).clamp_min(0)
    gmv = torch.expm1(gmv_log).clamp_min(0)

    positions = torch.arange(steps, device=x.device).float().unsqueeze(0)
    starts = (steps - lengths).float().unsqueeze(1)
    denom = (lengths.float() - 1).clamp_min(1.0).unsqueeze(1)

    relative_history_position = ((positions - starts) / denom).clamp(0, 1) * valid_mask
    days_to_cutoff = ((steps - 1 - positions) / max(steps - 1, 1)) * valid_mask

    derived = [
        valid_mask,
        relative_history_position,
        days_to_cutoff,
        (search_to_cart > 0).float(),
        (search_to_ord > 0).float(),
        (cat_to_cart > 0).float(),
        (cat_to_ord > 0).float(),
        safe_ratio(search_to_cart, searches),
        safe_ratio(search_to_ord, searches),
        safe_ratio(to_ord, to_cart),
        safe_ratio(gmv_search, gmv, 1.5),
    ]

    for values in [searches_log, to_cart_log, to_ord_log, gmv_log]:
        for window in [3, 7, 14, 30]:
            derived.append(causal_masked_mean(values, valid_mask, window))

    for window in [3, 7, 14, 30]:
        derived.append(causal_masked_mean(active, valid_mask, window))

    for values in [searches_log, to_ord_log, gmv_log]:
        derived.append(first_difference(values, valid_mask))

    return torch.cat([x, torch.stack(derived, dim=-1)], dim=-1)


def _raw_channel(x, feature_name):
    return torch.expm1(x[..., BASE_INDEX[feature_name]]).clamp_min(0)


def _window_raw_sum(x, feature_name, window):
    return _raw_channel(x, feature_name)[:, -window:].sum(dim=1)


def _slice_raw_sum(x, feature_name, start_from_end, end_from_end):
    values = _raw_channel(x, feature_name)
    left = x.shape[1] - start_from_end
    right = x.shape[1] - end_from_end if end_from_end > 0 else x.shape[1]
    return values[:, max(left, 0):max(right, 0)].sum(dim=1)


def _days_since_event(x, lengths, feature_name):
    values = _raw_channel(x, feature_name)
    valid = history_mask_from_lengths(lengths, x.shape[1])
    event = (values > 0) & valid
    positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0).expand_as(values)
    last = torch.where(event, positions, torch.full_like(positions, -1)).amax(dim=1)
    days = (x.shape[1] - 1 - last).float()
    return torch.where(last >= 0, days / x.shape[1], torch.full_like(days, 1.25))


def _purchase_periodicity(x, lengths):
    values = _raw_channel(x, "to_ord")
    valid = history_mask_from_lengths(lengths, x.shape[1])
    event = (values > 0) & valid

    positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0).expand_as(values)
    masked_pos = torch.where(event, positions, torch.full_like(positions, -1))
    top2 = torch.topk(masked_pos, k=2, dim=1).values
    last = top2[:, 0]
    previous = top2[:, 1]

    has_any = last >= 0
    has_two = previous >= 0

    days_since = torch.where(
        has_any,
        (x.shape[1] - 1 - last).float() / x.shape[1],
        torch.full_like(last.float(), 1.25),
    )
    last_gap = torch.where(
        has_two,
        (last - previous).float() / x.shape[1],
        torch.ones_like(last.float()),
    )
    event_rate = event.sum(dim=1).float() / lengths.float().clamp_min(1.0)
    phase = torch.where(
        has_two,
        days_since / last_gap.clamp_min(1.0 / x.shape[1]),
        torch.ones_like(days_since),
    ).clamp(0, 5)

    return [event_rate, days_since, last_gap, phase, has_two.float()]


def make_short_summary(x, lengths):
    windows = [1, 3, 7, 14, 30, 60, 90]
    volume_names = ["searches", "to_cart", "to_ord", "gmv", "gmv_search"]
    features = []

    for name in volume_names:
        for window in windows:
            features.append(torch.log1p(_window_raw_sum(x, name, window)))

    active = x[..., BASE_INDEX["active"]]
    for window in windows:
        exposure = lengths.clamp(max=window).float()
        features.append(active[:, -window:].sum(dim=1) / exposure.clamp_min(1.0))

    for name in volume_names:
        features.append(_days_since_event(x, lengths, name))

    for name in volume_names:
        values = _raw_channel(x, name)
        for window in [3, 7, 14, 30]:
            recent = values[:, -window:].sum(dim=1)
            previous = values[:, -2 * window:-window].sum(dim=1)
            features.append(torch.log1p(recent) - torch.log1p(previous))

    for name in volume_names:
        buckets = [
            _slice_raw_sum(x, name, 30, 0),
            _slice_raw_sum(x, name, 60, 30),
            _slice_raw_sum(x, name, 90, 60),
        ]
        bucket_logs = [torch.log1p(v) for v in buckets]
        features.extend(bucket_logs)
        features.append(bucket_logs[0] - bucket_logs[2])

    features.extend(_purchase_periodicity(x, lengths))
    features.append(lengths.float() / x.shape[1])

    return torch.stack(features, dim=1)


with torch.no_grad():
    dummy_x = torch.zeros(
        2,
        SEQ_LEN,
        len(BASE_SEQUENCE_FEATURES) + len(CALENDAR_FEATURES),
        dtype=torch.float32,
    )
    dummy_lengths = torch.tensor([SEQ_LEN, max(2, SEQ_LEN // 2)], dtype=torch.long)
    SEQ_INPUT_SIZE = int(make_sequence_features(dummy_x, dummy_lengths).shape[-1])
    SHORT_SUMMARY_SIZE = int(make_short_summary(dummy_x, dummy_lengths).shape[-1])

print("sequence features:", SEQ_INPUT_SIZE)
print("short summary:", SHORT_SUMMARY_SIZE)

sequence features: 53
short summary: 93


## 4. Static preprocessing

In [6]:
STATIC_INPUT_SIZE = 2 * len(STATIC_FEATURES) + len(STATIC_LOG_INDICES)


def augment_static_numpy(raw):
    raw = np.asarray(raw, dtype=np.float32)
    source = raw[:, STATIC_LOG_INDICES]
    logs = np.where(
        np.isfinite(source),
        np.log1p(np.clip(source, 0, None)),
        np.nan,
    ).astype(np.float32)
    missing = (~np.isfinite(raw)).astype(np.float32)
    return np.concatenate([raw, logs, missing], axis=1)


def fit_static_stats(cutoffs, chunk_size=65_536):
    sums = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    sums_sq = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    counts = np.zeros(STATIC_INPUT_SIZE, dtype=np.int64)

    for cutoff in cutoffs:
        raw_full = np.load(DATA_DIR / cutoff / "static.npy", mmap_mode="r")
        for start in range(0, len(raw_full), chunk_size):
            raw = np.asarray(raw_full[start:start + chunk_size], dtype=np.float32)[:, STATIC_INDICES]
            block = augment_static_numpy(raw)
            finite = np.isfinite(block)
            safe = np.where(finite, block, 0.0).astype(np.float64)
            sums += safe.sum(axis=0)
            sums_sq += (safe * safe).sum(axis=0)
            counts += finite.sum(axis=0)

    counts = np.maximum(counts, 1)
    mean = sums / counts
    std = np.sqrt(np.maximum(sums_sq / counts - mean * mean, 1e-6))
    return (
        torch.tensor(mean, dtype=torch.float32),
        torch.tensor(std, dtype=torch.float32),
    )


def normalize_static(raw, stats):
    source = raw[:, STATIC_LOG_INDICES]
    logs = torch.where(
        torch.isfinite(source),
        torch.log1p(source.clamp_min(0)),
        torch.nan,
    )
    augmented = torch.cat([raw, logs, (~torch.isfinite(raw)).float()], dim=1)

    mean, std = stats
    mean = mean.to(raw.device)
    std = std.to(raw.device)

    augmented = torch.where(torch.isfinite(augmented), augmented, mean)
    return (augmented - mean) / std


print("static features after augmentation:", STATIC_INPUT_SIZE)

static features after augmentation: 235


## 5. Exact JointHurdleLSTM architecture

In [7]:
def compact_left_padded(sequence, lengths):
    # Left padding переношу вправо перед packing.
    batch, steps, channels = sequence.shape
    positions = torch.arange(steps, device=sequence.device).unsqueeze(0).expand(batch, -1)
    starts = steps - lengths.unsqueeze(1)
    source_pos = (positions + starts).clamp(max=steps - 1)
    gathered = sequence.gather(
        1,
        source_pos.unsqueeze(-1).expand(-1, -1, channels),
    )
    valid = positions < lengths.unsqueeze(1)
    return gathered * valid.unsqueeze(-1)


class ResidualConvBlock(nn.Module):
    def __init__(self, channels, kernel_size=5, dropout=0.10):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, valid_mask):
        residual = x
        y = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        y = self.norm1(y)
        y = F.gelu(y)
        y = self.dropout(y)
        y = self.conv2(y.transpose(1, 2)).transpose(1, 2)
        y = self.norm2(y)
        y = F.gelu(y)
        y = self.dropout(y)
        return (residual + y) * valid_mask.unsqueeze(-1)


class JointHurdleLSTM(nn.Module):
    def __init__(
        self,
        n_users,
        user_embed_dim=0,
        d_model=80,
        hidden_size=112,
        num_layers=2,
        lstm_dropout=0.25,
        head_dropout=0.35,
        entity_dropout=0.60,
        use_conv_stem=False,
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.user_embed_dim = int(user_embed_dim)
        self.entity_dropout = float(entity_dropout)
        self.use_conv_stem = bool(use_conv_stem)

        self.sequence_projection = nn.Sequential(
            nn.Linear(SEQ_INPUT_SIZE, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.conv_stem = (
            ResidualConvBlock(d_model, kernel_size=5, dropout=0.10)
            if self.use_conv_stem
            else None
        )

        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.attention_score = nn.Sequential(
            nn.Linear(hidden_size * 2, 48),
            nn.Tanh(),
            nn.Linear(48, 1),
        )
        self.recency_attention_bias = nn.Parameter(torch.tensor(0.20))

        pooled_size = hidden_size * 2 * 4
        self.sequence_head = nn.Sequential(
            nn.LayerNorm(pooled_size),
            nn.Linear(pooled_size, 192),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(192, 144),
            nn.GELU(),
        )

        self.summary_head = nn.Sequential(
            nn.LayerNorm(SHORT_SUMMARY_SIZE),
            nn.Linear(SHORT_SUMMARY_SIZE, 96),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(96, 64),
            nn.GELU(),
        )

        self.static_head = nn.Sequential(
            nn.Linear(STATIC_INPUT_SIZE, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(192, 96),
            nn.GELU(),
            nn.Dropout(0.20),
        )

        if self.user_embed_dim > 0:
            self.user_embedding = nn.Embedding(n_users, self.user_embed_dim, max_norm=1.0)
            nn.init.normal_(self.user_embedding.weight, mean=0.0, std=0.015)
            self.user_scale_logit = nn.Parameter(torch.tensor(-3.0))
        else:
            self.user_embedding = None
            self.register_parameter("user_scale_logit", None)

        fusion_size = 144 + 64 + 96 + self.user_embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_size, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(192, 96),
            nn.GELU(),
            nn.Dropout(0.20),
        )

        self.gate_head = nn.Linear(96, 1)
        self.positive_head = nn.Linear(96, 1)
        self.direct_head = nn.Linear(96, 1)
        self.hurdle_mix_logit = nn.Parameter(torch.tensor(math.log(0.6 / 0.4)))

    def _user_repr(self, user_index, user_known):
        if self.user_embedding is None:
            return torch.empty(len(user_index), 0, device=user_index.device)

        emb = self.user_embedding(user_index)
        emb = emb * user_known.float().unsqueeze(1)

        if self.training and self.entity_dropout > 0:
            keep = (
                torch.rand((len(emb), 1), device=emb.device) >= self.entity_dropout
            ).float()
            emb = emb * keep / (1.0 - self.entity_dropout)

        scale = torch.sigmoid(self.user_scale_logit)
        return emb * scale

    def forward(self, sequence, static, user_index, user_known, history_length):
        short_summary = make_short_summary(sequence, history_length)

        sequence = make_sequence_features(sequence, history_length)
        sequence = compact_left_padded(sequence, history_length)

        positions = torch.arange(SEQ_LEN, device=sequence.device).unsqueeze(0)
        valid = positions < history_length.unsqueeze(1)
        valid_f = valid.float()

        sequence = self.sequence_projection(sequence) * valid_f.unsqueeze(-1)

        if self.conv_stem is not None:
            sequence = self.conv_stem(sequence, valid_f)

        packed = pack_padded_sequence(
            sequence,
            history_length.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, (hidden, _) = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=SEQ_LEN,
        )

        valid3 = valid.unsqueeze(-1)
        pooled_mean = (output * valid3).sum(dim=1) / history_length.float().unsqueeze(1)
        pooled_max = output.masked_fill(~valid3, float("-inf")).amax(dim=1)
        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)

        scores = self.attention_score(output).squeeze(-1)
        relative_pos = positions.float() / (
            history_length.unsqueeze(1).float() - 1
        ).clamp_min(1.0)
        scores = scores + self.recency_attention_bias * relative_pos
        scores = scores.masked_fill(~valid, -1e9)
        attn_weight = torch.softmax(scores, dim=1)
        pooled_attn = (output * attn_weight.unsqueeze(-1)).sum(dim=1)

        sequence_repr = self.sequence_head(torch.cat([
            last_hidden,
            pooled_mean,
            pooled_max,
            pooled_attn,
        ], dim=1))
        summary_repr = self.summary_head(short_summary)
        static_repr = self.static_head(static)
        user_repr = self._user_repr(user_index, user_known)

        fused = self.fusion(torch.cat([
            sequence_repr,
            summary_repr,
            static_repr,
            user_repr,
        ], dim=1))

        gate_logit = self.gate_head(fused).squeeze(1)
        positive_log = F.softplus(self.positive_head(fused).squeeze(1))
        direct_log = F.softplus(self.direct_head(fused).squeeze(1))

        gate_prob = torch.sigmoid(gate_logit)
        hurdle_log = gate_prob * positive_log
        hurdle_weight = torch.sigmoid(self.hurdle_mix_logit)

        pred_log = hurdle_weight * hurdle_log + (1.0 - hurdle_weight) * direct_log

        user_scale = (
            torch.sigmoid(self.user_scale_logit)
            if self.user_scale_logit is not None
            else torch.tensor(0.0, device=pred_log.device)
        )

        return {
            "pred_log": pred_log,
            "gate_logit": gate_logit,
            "gate_prob": gate_prob,
            "positive_log": positive_log,
            "direct_log": direct_log,
            "hurdle_log": hurdle_log,
            "hurdle_weight": hurdle_weight,
            "user_scale": user_scale,
        }


MODEL_KWARGS = {
    "n_users": N_USERS,
    "user_embed_dim": USER_EMBED_DIM if USE_USER_EMBEDDING else 0,
    "d_model": 80,
    "hidden_size": 112,
    "num_layers": 2,
    "lstm_dropout": 0.25,
    "head_dropout": 0.35,
    "entity_dropout": 0.60,
    "use_conv_stem": USE_CONV_STEM,
}

probe = JointHurdleLSTM(**MODEL_KWARGS)
print("parameters:", f"{sum(p.numel() for p in probe.parameters()):,}")
del probe

parameters: 851,216


## 6. Loss, optimizer, train и inference helpers

In [8]:
def unpack_batch(batch, with_target=True):
    if with_target:
        sequence, static, user_index, user_known, history_length, y = batch
        y = y.to(DEVICE, non_blocking=True)
    else:
        sequence, static, user_index, user_known, history_length = batch
        y = None

    return (
        sequence.to(DEVICE, non_blocking=True),
        static.to(DEVICE, non_blocking=True),
        user_index.to(DEVICE, non_blocking=True),
        user_known.to(DEVICE, non_blocking=True),
        history_length.to(DEVICE, non_blocking=True),
        y,
    )


def compute_joint_loss(out, y):
    target_log = torch.log1p(y)
    nonzero = (y > 0).float()

    main = F.mse_loss(out["pred_log"], target_log)
    gate = F.binary_cross_entropy_with_logits(out["gate_logit"], nonzero)

    positive_mask = y > 0
    if positive_mask.any():
        positive = F.mse_loss(
            out["positive_log"][positive_mask],
            target_log[positive_mask],
        )
    else:
        positive = out["positive_log"].sum() * 0.0

    direct = F.mse_loss(out["direct_log"], target_log)

    total = (
        main
        + AUX_GATE_WEIGHT * gate
        + AUX_POS_WEIGHT * positive
        + AUX_DIRECT_WEIGHT * direct
    )
    return total, {
        "main_mse": main.detach(),
        "gate_bce": gate.detach(),
        "positive_mse": positive.detach(),
        "direct_mse": direct.detach(),
    }


def make_optimizer(model):
    if model.user_embedding is None:
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )

    embed_ids = {id(p) for p in model.user_embedding.parameters()}
    embed_params = [p for p in model.parameters() if id(p) in embed_ids]
    main_params = [p for p in model.parameters() if id(p) not in embed_ids]

    return torch.optim.AdamW([
        {
            "params": main_params,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": embed_params,
            "lr": EMBED_LR,
            "weight_decay": EMBED_WEIGHT_DECAY,
        },
    ])


def new_grad_scaler():
    if not USE_AMP:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)


def train_one_epoch(model, loader, optimizer, static_stats, scaler=None):
    model.train()
    totals = {}
    total_n = 0

    for batch in loader:
        sequence, static, user_index, user_known, history_length, y = unpack_batch(batch)
        static = normalize_static(static, static_stats)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            out = model(sequence, static, user_index, user_known, history_length)
            loss, parts = compute_joint_loss(out, y)

        if scaler is not None and USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        n = len(y)
        totals["loss"] = totals.get("loss", 0.0) + float(loss.detach().cpu()) * n
        for key, value in parts.items():
            totals[key] = totals.get(key, 0.0) + float(value.cpu()) * n
        total_n += n

    return {k: v / max(total_n, 1) for k, v in totals.items()}


@torch.no_grad()
def predict_model(model, cutoff, static_stats, known_user_mask, with_target=True):
    dataset = HybridDataset(
        cutoff,
        with_target=with_target,
        known_user_mask=known_user_mask,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )

    model.eval()
    collected = {
        "pred_log": [],
        "gate_logit": [],
        "gate_prob": [],
        "positive_log": [],
        "direct_log": [],
        "hurdle_log": [],
    }
    targets = []

    for batch in loader:
        sequence, static, user_index, user_known, history_length, y = unpack_batch(
            batch,
            with_target=with_target,
        )
        static = normalize_static(static, static_stats)
        out = model(sequence, static, user_index, user_known, history_length)

        for key in collected:
            collected[key].append(out[key].detach().float().cpu().numpy())
        if with_target:
            targets.append(y.float().cpu().numpy())

    collected = {
        key: np.concatenate(value).astype(np.float32)
        for key, value in collected.items()
    }
    y = np.concatenate(targets).astype(np.float32) if with_target else None

    return np.asarray(dataset.users), collected, y


def _rmsle_from_log(target_log, pred_log):
    pred_log = np.clip(pred_log, 0, None)
    return float(np.sqrt(np.mean((target_log - pred_log) ** 2)))


def validation_metrics(y, pred, model):
    target_log = np.log1p(y)
    nonzero = (y > 0).astype(np.float32)

    p = np.clip(pred["gate_prob"], 1e-6, 1 - 1e-6)
    gate_bce = float(
        -np.mean(nonzero * np.log(p) + (1 - nonzero) * np.log(1 - p))
    )

    try:
        gate_auc = float(roc_auc_score(nonzero, p))
    except ValueError:
        gate_auc = float("nan")

    positive_mask = y > 0
    positive_rmse = float(np.sqrt(np.mean(
        (target_log[positive_mask] - pred["positive_log"][positive_mask]) ** 2
    )))

    return {
        "rmsle": _rmsle_from_log(target_log, pred["pred_log"]),
        "direct_rmsle": _rmsle_from_log(target_log, pred["direct_log"]),
        "hurdle_rmsle": _rmsle_from_log(target_log, pred["hurdle_log"]),
        "gate_bce": gate_bce,
        "gate_auc": gate_auc,
        "true_nonzero_rate": float(nonzero.mean()),
        "pred_nonzero_rate": float(p.mean()),
        "positive_rmse": positive_rmse,
        "hurdle_weight": float(torch.sigmoid(model.hurdle_mix_logit).detach().cpu()),
        "user_scale": (
            float(torch.sigmoid(model.user_scale_logit).detach().cpu())
            if model.user_scale_logit is not None
            else 0.0
        ),
    }

## 7. Читаем artifacts завершенного основного run

`config.json` является источником истины для:

- `BEST_EPOCH`;
- списка CV cutoff;
- model kwargs;
- scheduler horizon;
- training signature.

`cv_history.csv` нужен не для обучения, а чтобы после recovery сравнить RMSLE с тем, что было в исходном temporal CV.

In [9]:
OOF_MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_expanded_es"
OOF_RECOVERY_DIR = OOF_MODEL_DIR / "oof_recovery"
OOF_RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = OOF_MODEL_DIR / "config.json"
CV_HISTORY_PATH = OOF_MODEL_DIR / "cv_history.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Main LSTM config not found: {CONFIG_PATH}"
    )

if not CV_HISTORY_PATH.exists():
    raise FileNotFoundError(
        f"Main CV history not found: {CV_HISTORY_PATH}"
    )

main_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

BEST_EPOCH = int(main_config["best_epoch"])
CV_VALID_CUTOFFS = list(main_config["cv_valid_cutoffs"])
RECOVERY_T_MAX = int(main_config["scheduler_t_max"])

cv_history = pd.read_csv(CV_HISTORY_PATH)

if main_config["model_kwargs"] != MODEL_KWARGS:
    raise RuntimeError(
        "Notebook model kwargs differ from saved main-run config"
    )

print("BEST_EPOCH:", BEST_EPOCH)
print("CV folds:", CV_VALID_CUTOFFS)
print("scheduler T_max:", RECOVERY_T_MAX)
print("device:", DEVICE)
print("recovery dir:", OOF_RECOVERY_DIR)

BEST_EPOCH: 10
CV folds: ['2025-11-15', '2025-12-15', '2026-01-14']
scheduler T_max: 30
device: cuda
recovery dir: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/models/lstm_hurdle_v4_expanded_es/oof_recovery


## 8. Recovery helpers

Prediction artifact хранится как compressed `npz`.

Recovery checkpoint этого notebook отдельный от основного `cv_state`, поэтому мы ничего не перезаписываем в завершенном experiment directory.

После каждой восстановительной эпохи checkpoint обновляется атомарно. Если Colab оборвется, recovery можно продолжить.

In [10]:
def safe_torch_load_any(path, map_location=DEVICE):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def save_prediction_npz(path, users, y_true, pred):
    path.parent.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        path,
        user_id=users,
        y_true=y_true,
        pred_log=pred["pred_log"],
        gate_prob=pred["gate_prob"],
        positive_log=pred["positive_log"],
        direct_log=pred["direct_log"],
        hurdle_log=pred["hurdle_log"],
    )


def prediction_to_frame(path, cutoff, source):
    saved = np.load(path)

    frame = pd.DataFrame({
        "user_id": saved["user_id"],
        "cutoff_date": cutoff,
        "epoch": BEST_EPOCH,
        "y_true": saved["y_true"],
        "target_nonzero": (
            saved["y_true"] > 0
        ).astype(np.int8),
        "pred_log": saved["pred_log"],
        "gate_prob": saved["gate_prob"],
        "positive_log": saved["positive_log"],
        "direct_log": saved["direct_log"],
        "hurdle_log": saved["hurdle_log"],
        "source": source,
    })

    return frame


def frame_rmsle(frame):
    target_log = np.log1p(
        frame["y_true"].to_numpy(dtype=np.float64)
    )

    pred_log = frame[
        "pred_log"
    ].to_numpy(dtype=np.float64)

    return float(
        np.sqrt(
            np.mean(
                (target_log - pred_log) ** 2
            )
        )
    )


def save_recovery_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    static_stats,
):
    payload = {
        "epoch": int(epoch),
        "best_epoch": int(BEST_EPOCH),
        "model_kwargs": MODEL_KWARGS,
        "state_dict": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": (
            scaler.state_dict()
            if scaler is not None
            else None
        ),
        "static_mean": static_stats[0].cpu(),
        "static_std": static_stats[1].cpu(),
    }

    tmp = path.with_suffix(".tmp")
    torch.save(payload, tmp)
    tmp.replace(path)

## 9. Восстановление одного fold

Для missing prediction сначала проверяем original `resume.pt`.

Если original checkpoint уже **после** `BEST_EPOCH`, его нельзя использовать: SGD/AdamW необратимы, веса epoch 20 не позволяют получить веса epoch 10.

Тогда retrain идет только до `BEST_EPOCH`.

Если checkpoint ровно на selected epoch -- training вообще не нужен.

In [11]:
def recover_fold_prediction(
    fold_idx,
    valid_cutoff,
):
    valid_pos = LABELED_CUTOFFS.index(
        valid_cutoff
    )

    train_cutoffs = LABELED_CUTOFFS[
        :valid_pos
    ]

    original_fold_dir = (
        OOF_MODEL_DIR
        / "cv_state"
        / f"fold_{fold_idx}_{valid_cutoff}"
    )

    original_prediction = (
        original_fold_dir
        / f"prediction_epoch_{BEST_EPOCH:02d}.npz"
    )

    recovery_fold_dir = (
        OOF_RECOVERY_DIR
        / f"fold_{fold_idx}_{valid_cutoff}"
    )

    recovery_fold_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    recovered_prediction = (
        recovery_fold_dir
        / f"prediction_epoch_{BEST_EPOCH:02d}.npz"
    )

    recovery_resume = (
        recovery_fold_dir
        / "resume.pt"
    )

    # 1. Самый дешевый путь -- exact prediction основного run.
    if original_prediction.exists():
        return (
            original_prediction,
            "main_cv_prediction",
        )

    # 2. Уже восстановили раньше.
    if recovered_prediction.exists():
        return (
            recovered_prediction,
            "recovery_cache",
        )

    set_seed(RANDOM_STATE)

    static_stats = fit_static_stats(
        train_cutoffs
    )

    known_user_mask = (
        known_user_mask_from_cutoffs(
            train_cutoffs
        )
    )

    train_loader = make_loader(
        train_cutoffs,
        shuffle=True,
        known_user_mask=known_user_mask,
    )

    model = JointHurdleLSTM(
        **MODEL_KWARGS
    ).to(DEVICE)

    optimizer = make_optimizer(model)
    scaler = new_grad_scaler()

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=RECOVERY_T_MAX,
            eta_min=2e-5,
        )
    )

    start_epoch = 1
    resume_source = "fresh_retrain"

    # Сначала checkpoint, созданный этим recovery notebook.
    if recovery_resume.exists():
        checkpoint = safe_torch_load_any(
            recovery_resume
        )

        if (
            int(checkpoint.get("best_epoch", -1))
            == BEST_EPOCH
            and checkpoint.get("model_kwargs")
            == MODEL_KWARGS
            and int(checkpoint["epoch"])
            <= BEST_EPOCH
        ):
            model.load_state_dict(
                checkpoint["state_dict"],
                strict=True,
            )

            optimizer.load_state_dict(
                checkpoint["optimizer_state_dict"]
            )

            scheduler.load_state_dict(
                checkpoint["scheduler_state_dict"]
            )

            if (
                scaler is not None
                and checkpoint.get("scaler_state_dict")
                is not None
            ):
                scaler.load_state_dict(
                    checkpoint["scaler_state_dict"]
                )

            static_stats = (
                checkpoint["static_mean"].cpu(),
                checkpoint["static_std"].cpu(),
            )

            start_epoch = (
                int(checkpoint["epoch"])
                + 1
            )

            resume_source = "recovery_resume"

    # Если recovery checkpoint нет, можно использовать original resume,
    # но только если он не прошел дальше selected epoch.
    elif (
        original_fold_dir / "resume.pt"
    ).exists():
        checkpoint = safe_torch_load_any(
            original_fold_dir / "resume.pt"
        )

        original_epoch = int(
            checkpoint["epoch"]
        )

        if (
            checkpoint.get("model_kwargs")
            == MODEL_KWARGS
            and original_epoch <= BEST_EPOCH
        ):
            model.load_state_dict(
                checkpoint["state_dict"],
                strict=True,
            )

            optimizer.load_state_dict(
                checkpoint["optimizer_state_dict"]
            )

            scheduler.load_state_dict(
                checkpoint["scheduler_state_dict"]
            )

            if (
                scaler is not None
                and checkpoint.get("scaler_state_dict")
                is not None
            ):
                scaler.load_state_dict(
                    checkpoint["scaler_state_dict"]
                )

            static_stats = (
                checkpoint["static_mean"].cpu(),
                checkpoint["static_std"].cpu(),
            )

            start_epoch = original_epoch + 1
            resume_source = "main_cv_resume"

        else:
            print(
                f"{valid_cutoff}: original resume is epoch "
                f"{original_epoch} > BEST_EPOCH={BEST_EPOCH} "
                "-- cannot rewind, retrain from scratch"
            )

    for epoch in range(
        start_epoch,
        BEST_EPOCH + 1,
    ):
        metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            static_stats,
            scaler,
        )

        scheduler.step()

        save_recovery_checkpoint(
            recovery_resume,
            epoch,
            model,
            optimizer,
            scheduler,
            scaler,
            static_stats,
        )

        print(
            f"{valid_cutoff} | "
            f"epoch {epoch:02d}/{BEST_EPOCH:02d} | "
            f"train_RMSE="
            f"{np.sqrt(max(metrics['main_mse'], 0.0)):.5f}"
        )

    users, pred, y_true = predict_model(
        model,
        valid_cutoff,
        static_stats,
        known_user_mask,
        with_target=True,
    )

    save_prediction_npz(
        recovered_prediction,
        users,
        y_true,
        pred,
    )

    del model, optimizer, train_loader

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return (
        recovered_prediction,
        resume_source,
    )

## 10. Собираем полный OOF

После этой клетки на каждый validation cutoff будет ровно один selected-epoch prediction.

В таблицу также пишется `source`, чтобы было видно, какой fold:

- взят напрямую из основного run;
- взят из recovery cache;
- восстановлен из checkpoint;
- переобучен с нуля.

In [12]:
oof_parts = []
diagnostic_rows = []

for fold_idx, valid_cutoff in enumerate(
    CV_VALID_CUTOFFS,
    start=1,
):
    print("\n" + "=" * 90)
    print(
        f"OOF fold {fold_idx}/{len(CV_VALID_CUTOFFS)} "
        f"| valid={valid_cutoff}"
    )

    prediction_path, source = (
        recover_fold_prediction(
            fold_idx,
            valid_cutoff,
        )
    )

    frame = prediction_to_frame(
        prediction_path,
        valid_cutoff,
        source,
    )

    observed_rmsle = frame_rmsle(frame)

    stored = cv_history[
        (cv_history["valid_cutoff"] == valid_cutoff)
        & (cv_history["epoch"] == BEST_EPOCH)
    ]

    stored_rmsle = (
        float(stored.iloc[0]["rmsle"])
        if len(stored) == 1
        else np.nan
    )

    diagnostic_rows.append({
        "fold": fold_idx,
        "valid_cutoff": valid_cutoff,
        "source": source,
        "rows": len(frame),
        "stored_cv_rmsle": stored_rmsle,
        "recovered_rmsle": observed_rmsle,
        "delta": (
            observed_rmsle - stored_rmsle
            if np.isfinite(stored_rmsle)
            else np.nan
        ),
    })

    oof_parts.append(frame)

oof_best = pd.concat(
    oof_parts,
    ignore_index=True,
)

diagnostics = pd.DataFrame(
    diagnostic_rows
)

display(diagnostics)

OOF_PATH = (
    OOF_MODEL_DIR
    / "oof_best_epoch.parquet"
)

OOF_DIAGNOSTICS_PATH = (
    OOF_MODEL_DIR
    / "oof_best_epoch_diagnostics.csv"
)

oof_best.to_parquet(
    OOF_PATH,
    index=False,
)

diagnostics.to_csv(
    OOF_DIAGNOSTICS_PATH,
    index=False,
)

print()
print("saved:", OOF_PATH)
print("saved:", OOF_DIAGNOSTICS_PATH)
print("rows:", len(oof_best))
print(
    "cutoffs:",
    oof_best["cutoff_date"].value_counts().sort_index().to_dict(),
)


OOF fold 1/3 | valid=2025-11-15
2025-11-15: original resume is epoch 20 > BEST_EPOCH=10 -- cannot rewind, retrain from scratch
2025-11-15 | epoch 01/10 | train_RMSE=2.17683
2025-11-15 | epoch 02/10 | train_RMSE=1.77885
2025-11-15 | epoch 03/10 | train_RMSE=1.76355
2025-11-15 | epoch 04/10 | train_RMSE=1.75724
2025-11-15 | epoch 05/10 | train_RMSE=1.75398
2025-11-15 | epoch 06/10 | train_RMSE=1.75078
2025-11-15 | epoch 07/10 | train_RMSE=1.74924
2025-11-15 | epoch 08/10 | train_RMSE=1.74765
2025-11-15 | epoch 09/10 | train_RMSE=1.74648
2025-11-15 | epoch 10/10 | train_RMSE=1.74522

OOF fold 2/3 | valid=2025-12-15
2025-12-15: original resume is epoch 12 > BEST_EPOCH=10 -- cannot rewind, retrain from scratch
2025-12-15 | epoch 01/10 | train_RMSE=2.14110
2025-12-15 | epoch 02/10 | train_RMSE=1.77463
2025-12-15 | epoch 03/10 | train_RMSE=1.76183
2025-12-15 | epoch 04/10 | train_RMSE=1.75575
2025-12-15 | epoch 05/10 | train_RMSE=1.75237
2025-12-15 | epoch 06/10 | train_RMSE=1.75017
2025-12-

,fold,valid_cutoff,source,rows,stored_cv_rmsle,recovered_rmsle,delta
0,1,2025-11-15,fresh_retrain,244983,1.731923,1.732471,0.000548
1,2,2025-12-15,fresh_retrain,250000,1.737679,1.744644,0.006966
2,3,2026-01-14,fresh_retrain,250000,1.677522,1.674130,-0.003392



saved: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/models/lstm_hurdle_v4_expanded_es/oof_best_epoch.parquet
saved: /content/drive/MyDrive/Colab-Notebooks/E-CUP-2026/models/lstm_hurdle_v4_expanded_es/oof_best_epoch_diagnostics.csv
rows: 744983
cutoffs: {'2025-11-15': 244983, '2025-12-15': 250000, '2026-01-14': 250000}


## 11. Sanity checks

Проверяем, что OOF пригоден для последующей calibration / meta-analysis:

- нет duplicate `user_id x cutoff`;
- все predictions конечные;
- `gate_prob` лежит в `[0,1]`;
- каждый cutoff содержит полный validation user set;
- RMSLE по fold посчитан и сравнен с исходной `cv_history`.

Если fold был retrain на другом backend, небольшой `delta` относительно исходной CV допустим. Большой delta -- повод не смешивать такой recovery OOF с original fold без дополнительной проверки.

In [13]:
required_columns = [
    "user_id",
    "cutoff_date",
    "y_true",
    "target_nonzero",
    "pred_log",
    "gate_prob",
    "positive_log",
    "direct_log",
    "hurdle_log",
]

if oof_best.duplicated(
    ["user_id", "cutoff_date"]
).any():
    raise RuntimeError(
        "Duplicate user_id x cutoff in OOF"
    )

for column in [
    "y_true",
    "pred_log",
    "gate_prob",
    "positive_log",
    "direct_log",
    "hurdle_log",
]:
    if not np.isfinite(
        oof_best[column].to_numpy()
    ).all():
        raise RuntimeError(
            f"Non-finite values in {column}"
        )

if not (
    oof_best["gate_prob"].between(
        0.0,
        1.0,
    ).all()
):
    raise RuntimeError(
        "gate_prob outside [0, 1]"
    )

print("OOF SANITY CHECK: PASSED")
display(
    oof_best[required_columns + ["source"]].head()
)

OOF SANITY CHECK: PASSED


,user_id,cutoff_date,y_true,target_nonzero,pred_log,gate_prob,positive_log,direct_log,hurdle_log,source
0,2,2025-11-15,0.000000,0,0.500319,0.140711,3.543288,0.502829,0.498578,fresh_retrain
1,7,2025-11-15,181.124603,1,3.024007,0.799439,3.863985,2.930260,3.089021,fresh_retrain
2,15,2025-11-15,133.692444,1,0.656375,0.187998,3.528020,0.646447,0.663260,fresh_retrain
3,18,2025-11-15,25.739683,1,4.552717,0.964007,4.790531,4.458429,4.618107,fresh_retrain
4,23,2025-11-15,0.000000,0,0.548686,0.156526,3.521180,0.545122,0.551157,fresh_retrain
